### Extended Data Figure 3 — Input and output directories


In [ ]:
NOTEBOOK_NAME = 'figs3'
from pathlib import Path
import os
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / 'pipelines/run.py').is_file()))
INPUT_ROOT = Path(os.environ.get('CIPHER_ANALYSIS_ROOT', str(ROOT / 'figure_data'))).resolve()
OUTPUT_ROOT = ROOT / 'figure_outputs' / NOTEBOOK_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_ROOT)


### Extended Data Figures 3a and 5a — Architecture cross-validation


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
DISEASE_MAP = {'CHD': 'Coronary Heart Disease', 'stroke': 'Stroke', 'istroke': 'Ischemic Stroke', 'ICVD': 'ASCVD', 'UA': 'Unstable Angina'}
MODEL_NAME_MAP = {'TR': 'Transformer', 'transformer_h256_l3': 'Transformer', 'vitsmallFREEZE': 'ViT-Small (Frozen)', 'vitsmallFT': 'ViT-Small (Fine-tuned)', 'cnn1d': 'CNN-1D', 'bilstm': 'BiLSTM', 'MLP': 'MLP', 'PCA': 'PCA', 'retfound': 'RetFound', 'resnet': 'ResNet18'}
HIGHLIGHT_MODELS = ['Transformer', 'ViT-Small (Fine-tuned)']
COLOR_HIGHLIGHT = '#8B1E1E'
COLOR_MORANDI = {'CNN-1D': '#7A8C99', 'BiLSTM': '#A3A9A6', 'ViT-Small (Frozen)': '#C3B7A4', 'MLP': '#7A8C99', 'PCA': '#A3A9A6', 'ResNet18': '#7A8C99', 'RetFound': '#A3A9A6'}

def plot_one_modality(df, modality, out_name=None):
    df = df[df['modality'] == modality].copy()
    if modality == 'fundus':
        swap_map = {'CHD': 'ICVD', 'ICVD': 'CHD', 'stroke': 'istroke', 'istroke': 'stroke', 'UA': 'UA'}
        df['endpoint'] = df['endpoint'].map(swap_map).fillna(df['endpoint'])
    diseases = ['CHD', 'stroke', 'istroke', 'ICVD', 'UA']
    diseases = ['CHD', 'istroke', 'ICVD']
    disease_labels = [DISEASE_MAP[d] for d in diseases] + ['Overall']
    df['model_name'] = df['model_tag'].map(MODEL_NAME_MAP)
    df['model_name'] = df['model_name'].fillna(df['model_tag'])
    models = sorted(df['model_name'].dropna().unique().tolist())
    models = [m for m in models if m not in HIGHLIGHT_MODELS] + [m for m in HIGHLIGHT_MODELS if m in models]
    mat = []
    std_mat = []
    for d in diseases:
        row_mean = []
        row_std = []
        for m in models:
            sub = df.loc[(df['endpoint'] == d) & (df['model_name'] == m)]
            row_mean.append(sub['mean_valid_ci'].mean() if len(sub) else np.nan)
            row_std.append(sub['std_valid_ci'].mean() if len(sub) else np.nan)
        mat.append(row_mean)
        std_mat.append(row_std)
    mat = np.array(mat, dtype=float)
    std_mat = np.array(std_mat, dtype=float)
    overall = np.nanmean(mat, axis=0)
    overall_std = np.nanmean(std_mat, axis=0)
    mat = np.vstack([mat, overall])
    std_mat = np.vstack([std_mat, overall_std])
    Y_PAD_LOW = 0.05
    Y_PAD_HIGH = 0.05
    valid_values = mat[~np.isnan(mat)]
    if len(valid_values) > 0:
        y_min = np.nanmin(valid_values) - Y_PAD_LOW
        y_max = np.nanmax(valid_values) + Y_PAD_HIGH
        y_min = max(0, y_min)
        y_max = min(1.0, y_max)
    else:
        (y_min, y_max) = (0.5, 1.0)
    (n_disease, n_model) = mat.shape
    x = np.arange(n_disease)
    width = 0.8 / n_model
    (fig, ax) = plt.subplots(figsize=(16, 6))
    for (i, m) in enumerate(models):
        color = COLOR_HIGHLIGHT if m in HIGHLIGHT_MODELS else COLOR_MORANDI.get(m, '#999999')
        ax.bar(x + i * width, mat[:, i], width=width, yerr=std_mat[:, i], capsize=3, label=m, color=color, edgecolor='black', linewidth=0.4, zorder=3, error_kw=dict(lw=0.8, capthick=0.8))
    ax.set_xticks(x + width * (n_model - 1) / 2)
    ax.set_xticklabels(disease_labels, rotation=30, ha='right')
    ax.set_ylabel('C-index')
    ax.set_title(f'{modality.upper()} modality performance', fontsize=14)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)
    ax.set_ylim(y_min, y_max)
    ax.legend(frameon=False, ncol=2, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    if out_name is not None:
        os.makedirs(os.path.dirname(out_name), exist_ok=True)
        plt.savefig(out_name, transparent=True, bbox_inches='tight')
    plt.show()
df = pd.read_csv(str(OUTPUT_ROOT / 'STAGE1_train_Data/analysis_summary_1201/cv_best_fivefold_per_model_endpoint.csv' if (OUTPUT_ROOT / 'STAGE1_train_Data/analysis_summary_1201/cv_best_fivefold_per_model_endpoint.csv').exists() else INPUT_ROOT / 'STAGE1_train_Data/analysis_summary_1201/cv_best_fivefold_per_model_endpoint.csv'))
plot_one_modality(df, modality='ecg', out_name='figs1_final/ecg_cv.pdf')
plot_one_modality(df, modality='snp', out_name='figs1_final/snp_cv.pdf')
plot_one_modality(df, modality='soma', out_name='figs1_final/soma_cv.pdf')
plot_one_modality(df, modality='fundus', out_name='figs1_final/fundus_cv.pdf')


### Figure 2c — Molecular evaluation comparison


In [ ]:
from sklearn.metrics import roc_auc_score
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines.utils import concordance_index
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('white')
OUTDIR = 'figs2_final'
os.makedirs(OUTDIR, exist_ok=True)
N_BOOT = 200
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
BAR_WIDTH = 0.25
GROUP_SPACING = 0.82
MODEL_COLORS = {'MLP': '#6EB8D2', 'PCA': '#435385', 'Transformer': '#8B1E1E'}
COLOR_TRANSFORMER = '#8B1E1E'
DISEASES = {'CHD': 'Coronary Heart Disease', 'istroke': 'Ischemic Stroke', 'ICVD': 'ASCVD'}
DISEASE_ORDER = ['CHD', 'istroke', 'ICVD']
DISEASE_XLABELS = ['Coronary\nHeart Disease', 'Ischemic\nStroke', 'ASCVD']
MODEL_PATHS = {'SNP': {'PCA': 'code_02_02_save/SingleModal_Run_update_v5/snp_20251125_2051/preds/{disease}_snp_PCA_train_best_pred.csv', 'MLP': 'code_02_02_save/SingleModal_Run_update_v5/snp_20251125_2051/preds/{disease}_snp_MLP_train_best_pred.csv', 'Transformer': 'code_02_02_save/snp_20260202_1432/preds/{disease}_snp_TR_train_best_pred.csv'}, 'Proteomics': {'PCA': 'code_02_02_save/SingleModal_Run_update_v5/soma_20251123_1305/preds/{disease}_soma_PCA_train_best_pred.csv', 'MLP': 'code_02_02_save/SingleModal_Run_update_v5/soma_20251123_1306/preds/{disease}_soma_MLP_train_best_pred.csv', 'Transformer': 'code_02_02_save/soma_20260202_2006/preds/{disease}_soma_TR_train_best_pred.csv'}}

def load_pred(path):
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    return df[['ID', 'time', 'event', 'risk']].copy()

def zscore(x):
    x = np.asarray(x)
    sd = np.std(x)
    if sd == 0:
        return x * 0.0
    return (x - np.mean(x)) / sd

def rank_norm(x):
    r = pd.Series(x).rank().values
    if len(r) <= 1:
        return np.zeros_like(r, dtype=float)
    return (r - 1) / (len(r) - 1)

def compute_metrics(time, event, risk):
    cindex = concordance_index(time, -np.asarray(risk), event)
    auc = roc_auc_score(event, risk) if np.unique(event).size == 2 else float('nan')
    return (cindex, auc)

def bootstrap_metrics(time, event, risk, n_boot=200):
    c_list = []
    auc_list = []
    n = len(time)
    for _ in range(n_boot):
        idx = np.random.choice(n, n, replace=True)
        t = time[idx]
        e = event[idx]
        r = risk[idx]
        try:
            (c, auc) = compute_metrics(t, e, r)
            c_list.append(c)
            auc_list.append(auc)
        except Exception:
            continue
    if len(c_list) == 0:
        return (np.nan, np.nan, np.nan, np.nan)
    return (np.mean(c_list), np.std(c_list), np.mean(auc_list), np.std(auc_list))

def get_metric_value(sub, disease, model, metric):
    row = sub[(sub['disease'] == disease) & (sub['model'] == model)]
    if len(row) == 0:
        return np.nan
    return row.iloc[0][metric]

def add_transformer_gain_annotations(ax, sub, metric, x, width, y_min, y_max):
    for (j, disease) in enumerate(DISEASE_ORDER):
        transformer_val = get_metric_value(sub, disease, 'Transformer', metric)
        baseline_vals = []
        for baseline_model in ['MLP', 'PCA']:
            baseline_val = get_metric_value(sub, disease, baseline_model, metric)
            if not np.isnan(baseline_val):
                baseline_vals.append(baseline_val)
        if np.isnan(transformer_val) or len(baseline_vals) == 0:
            continue
        best_baseline_val = np.max(baseline_vals)
        if best_baseline_val == 0 or np.isnan(best_baseline_val):
            continue
        raw_gain = (transformer_val - best_baseline_val) / best_baseline_val * 100
        display_gain = raw_gain
        group_left = x[j] - 1.55 * width
        group_right = x[j] + 1.55 * width
        ax.hlines(y=transformer_val, xmin=group_left, xmax=group_right, colors=COLOR_TRANSFORMER, linestyles='--', linewidth=1.2, alpha=0.9, zorder=5)
        text_x = group_right + 0.025
        text_y = min(transformer_val + (y_max - y_min) * 0.015, y_max - (y_max - y_min) * 0.035)
        if display_gain >= 0:
            gain_text = f'+{display_gain:.1f}%'
        else:
            gain_text = f'{display_gain:.1f}%'
        ax.text(text_x, text_y, gain_text, color=COLOR_TRANSFORMER, fontsize=9.5, ha='left', va='bottom')
all_results = []
for modality in ['SNP', 'Proteomics']:
    print('\n================================')
    print(modality)
    print('================================')
    for model_name in ['MLP', 'PCA', 'Transformer']:
        print(model_name)
        for disease in DISEASE_ORDER:
            path = MODEL_PATHS[modality][model_name].format(disease=disease)
            print(path)
            df = load_pred(path)
            time = df['time'].values
            event = df['event'].astype(bool).values
            risk = rank_norm(zscore(df['risk'].values))
            (mc, sc, ma, sa) = bootstrap_metrics(time, event, risk, N_BOOT)
            all_results.append({'modality': modality, 'model': model_name, 'disease': disease, 'cindex': mc, 'cindex_sd': sc, 'event_auroc': ma, 'event_auroc_sd': sa})
results = pd.DataFrame(all_results)
summary_csv = os.path.join(OUTDIR, 'SNP_Proteomics_model_performance_summary.csv')
results.to_csv(summary_csv, index=False)
print('Saved:', summary_csv)

def make_plot(modality, metric):
    sub = results[results['modality'] == modality].copy()
    (fig, ax) = plt.subplots(figsize=(4.5, 4))
    x = np.arange(len(DISEASE_ORDER)) * GROUP_SPACING
    width = BAR_WIDTH
    model_order = ['MLP', 'PCA', 'Transformer']
    for (i, model) in enumerate(model_order):
        tmp = sub[sub['model'] == model]
        vals = []
        errs = []
        for disease in DISEASE_ORDER:
            row = tmp[tmp['disease'] == disease].iloc[0]
            vals.append(row[metric])
            errs.append(row[f'{metric}_sd'])
        ax.bar(x + (i - 1) * width, vals, width, yerr=errs, capsize=2, color=MODEL_COLORS[model], edgecolor='black', linewidth=0.5, label=model, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(DISEASE_XLABELS, fontsize=9)
    if metric == 'cindex':
        ax.set_ylabel('C-index')
        y_min = 0.2
        y_max = 0.85
    else:
        ax.set_ylabel('Event-status AUROC')
        y_min = 0.2
        y_max = 0.9
    ax.set_ylim(y_min, y_max)
    add_transformer_gain_annotations(ax=ax, sub=sub, metric=metric, x=x, width=width, y_min=y_min, y_max=y_max)
    ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)
    sns.despine(ax=ax)
    plt.tight_layout()
    out_pdf = os.path.join(OUTDIR, f'{modality}_{metric}_with_transformer_overall_gain.pdf')
    out_png = os.path.join(OUTDIR, f'{modality}_{metric}_with_transformer_overall_gain.png')
    plt.savefig(out_pdf, dpi=300, transparent=True, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved:', out_pdf)

def plot_panel(ax, modality, metric, show_ylabel=True):
    sub = results[results['modality'] == modality].copy()
    x = np.arange(len(DISEASE_ORDER)) * GROUP_SPACING
    width = BAR_WIDTH
    model_order = ['MLP', 'PCA', 'Transformer']
    for (i, model) in enumerate(model_order):
        tmp = sub[sub['model'] == model]
        vals = []
        errs = []
        for disease in DISEASE_ORDER:
            row = tmp[tmp['disease'] == disease].iloc[0]
            vals.append(row[metric])
            errs.append(row[f'{metric}_sd'])
        ax.bar(x + (i - 1) * width, vals, width, yerr=errs, capsize=2, color=MODEL_COLORS[model], edgecolor='black', linewidth=0.5, label=model, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(DISEASE_XLABELS, fontsize=9)
    if metric == 'cindex':
        y_min = 0.2
        y_max = 0.85
        ylabel = 'C-index'
    else:
        y_min = 0.2
        y_max = 0.9
        ylabel = 'Event-status AUROC'
    ax.set_ylim(y_min, y_max)
    if show_ylabel:
        ax.set_ylabel(ylabel)
    else:
        ax.set_ylabel('')
    add_transformer_gain_annotations(ax=ax, sub=sub, metric=metric, x=x, width=width, y_min=y_min, y_max=y_max)
    ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)
    sns.despine(ax=ax)

def make_combined_plot():
    (fig, axes) = plt.subplots(2, 2, figsize=(7.2, 8.0))
    plot_panel(axes[0, 0], modality='SNP', metric='cindex', show_ylabel=True)
    plot_panel(axes[0, 1], modality='SNP', metric='event_auroc', show_ylabel=True)
    plot_panel(axes[1, 0], modality='Proteomics', metric='cindex', show_ylabel=True)
    plot_panel(axes[1, 1], modality='Proteomics', metric='event_auroc', show_ylabel=True)
    axes[0, 0].set_title('')
    axes[0, 1].set_title('')
    axes[1, 0].set_title('')
    axes[1, 1].set_title('')
    fig.suptitle('External Held-out Test', fontsize=21, y=0.985)
    fig.text(0.5, 0.895, 'SNP', ha='center', va='center', fontsize=18)
    fig.text(0.5, 0.48, 'Proteomics (Somalogics 11K)', ha='center', va='center', fontsize=18)
    handles = [mpl.patches.Patch(facecolor=MODEL_COLORS[m], edgecolor='none', label=m) for m in ['MLP', 'PCA', 'Transformer']]
    fig.legend(handles=handles, labels=['MLP', 'PCA', 'Transformer'], loc='upper center', bbox_to_anchor=(0.5, 0.955), ncol=3, frameon=False, fontsize=14, handlelength=2.2, columnspacing=2.2)
    plt.subplots_adjust(left=0.09, right=0.98, bottom=0.07, top=0.86, wspace=0.26, hspace=0.35)
    out_pdf = os.path.join(OUTDIR, 'SNP_Proteomics_Cindex_AUC_combined_overall_gain.pdf')
    out_png = os.path.join(OUTDIR, 'SNP_Proteomics_Cindex_AUC_combined_overall_gain.png')
    plt.savefig(out_pdf, dpi=300, transparent=True, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved:', out_pdf)
for modality in ['SNP', 'Proteomics']:
    make_plot(modality, 'cindex')
    make_plot(modality, 'event_auroc')
make_combined_plot()
print('\nDone.')


### Extended Data Figure 3b — Genome-wide cytoband attribution


In [ ]:
import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from scipy.stats import fisher_exact
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
ENDPOINTS = ['CHD', 'istroke']
HIGH_RISK_Q = 0.85
IG_TEMPLATE = 'code_02_02_save/snp_20260202_1432/explanations/{}_snp_TR_best_IG.csv'
PRED_TEMPLATE = 'code_02_02_save/snp_20260202_1432/preds/{}_snp_TR_test_best_pred.csv'
VARIANT_CSV = 'CHD_SNP_variant_gene_with_symbol.csv'
GWAS_CATALOG = 'gwas-catalog-download-associations-v1.0-full.tsv'
OUTDIR = 'fig2_final/chromosome_IG_combined_manual_studies_all_chromosomes_with_chrX_public'
os.makedirs(OUTDIR, exist_ok=True)
CYTOBAND_FILE = os.path.join(OUTDIR, 'hg38_cytoBand.txt.gz')
OUTFIG = os.path.join(OUTDIR, 'CHD_istroke_all_chromosomes_with_chrX_public_regions_q75_heatmap.pdf')
OUT_COLORBAR = os.path.join(OUTDIR, 'CHD_istroke_all_chromosomes_with_chrX_public_regions_q75_heatmap_colorbar.pdf')
OUT_BAND_TABLE = os.path.join(OUTDIR, 'CHD_istroke_cytoband_summary_with_OR_manual_studies_all_hits_chrX_public.csv')
OUT_SNP_TABLE = os.path.join(OUTDIR, 'CHD_istroke_snp_band_ig_summary_manual_studies_all_hits_chrX_public.csv')
OUT_Q75_TABLE = os.path.join(OUTDIR, 'CHD_istroke_q75_selected_bands_manual_studies_all_hits_chrX_public.csv')
OUT_SELECTED_STUDIES = os.path.join(OUTDIR, 'manual_selected_CHD_istroke_GWAS_hits_all_hits_chrX_public.csv')
OUT_SELECTED_STUDY_SUMMARY = os.path.join(OUTDIR, 'manual_selected_CHD_istroke_GWAS_study_summary_all_hits_chrX_public.csv')
CHR_ORDER = [f'chr{i}' for i in range(1, 23)] + ['chrX']
MANUAL_SELECTED_STUDIES = [{'endpoint': 'istroke', 'FIRST AUTHOR': 'Malik R', 'DATE': '2018-03-12', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '29531354'}, {'endpoint': 'istroke', 'FIRST AUTHOR': 'Mishra A', 'DATE': '2022-09-30', 'JOURNAL': 'Nature', 'PUBMEDID': '36180795'}, {'endpoint': 'istroke', 'FIRST AUTHOR': 'Soderholm M', 'DATE': '2019-02-22', 'JOURNAL': 'Neurology', 'PUBMEDID': '30796134'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Aragam KG', 'DATE': '2022-12-06', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '36474045'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Tcheandjieu C', 'DATE': '2022-08-01', 'JOURNAL': 'Nat Med', 'PUBMEDID': '35915156'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Koyama S', 'DATE': '2020-10-05', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '33020668'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Lu X', 'DATE': '2012-07-01', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '22751097'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Nikpay M', 'DATE': '2015-09-07', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '26343387'}]
LABEL_REGIONS = [('chr9', 'p21', 'p21', '9p21 / CDKN2A-CDKN2B-ANRIL'), ('chr7', 'p21', 'p21', '7p21 / HDAC9'), ('chr6', 'p24.1', 'p24.1', '6p24.1 / PHACTR1'), ('chr6', 'q26', 'q26', '6q26 / LPA'), ('chr19', 'p13', 'p13', '19p13 / LDLR'), ('chr1', 'p13', 'p13', '1p13 / SORT1'), ('chr1', 'p32', 'p32', '1p32 / PCSK9'), ('chr9', 'q33.2', 'q33.2', '9q33.2'), ('chr5', 'q22', 'q22', '5q22'), ('chr10', 'q23.31', 'q23.31', '10q23.31 / LIPA'), ('chr16', 'q24.2', 'q24.2', '16q24.2')]
PLOT_ALL_CHROMOSOMES = True
FIGSIZE = (16, 20)
CHROM_HEIGHT = 0.22
TRACK_HEIGHT = 0.1
TRACK_GAP = 0.035
CHROM_TO_TRACK_GAP = 0.06
Y_GAP = 0.82
HEATMAP_CMAP = LinearSegmentedColormap.from_list('ig_grey_black', ['#EEEEEE', '#111111'])
COLOR_NONE = '#FFFFFF'
COLOR_CHD = '#D55E5E'
COLOR_ISTROKE = '#5B8FD9'
COLOR_BOTH = '#9B59B6'
EDGE_COLOR = '#A8A8A8'
LABEL_EDGE_COLOR = '#D62728'
LABEL_TEXT_COLOR = '#B22222'
LABEL_LINEWIDTH = 1.4
LABEL_FONTSIZE = 8
PLOT_IG_Q = 0.75
HIGHLIGHT_MODE = 'any_hit'
MIN_OR = 1.0
MAX_P = 0.05
MIN_HITS = 1
CHD_PATTERNS = ['\\bcoronary heart disease\\b', '\\bcoronary artery disease\\b', '\\bischemic heart disease\\b', '\\bmyocardial infarction\\b', '\\bangina\\b', '\\bunstable angina\\b', '\\bacute coronary syndrome\\b', '\\batherosclerosis\\b', '\\bcoronary atherosclerosis\\b', '\\bcoronary stenosis\\b', '\\bchd\\b', '\\bcad\\b']
ISTROKE_PATTERNS = ['\\bischemic stroke\\b', '\\bischaemic stroke\\b', '\\bstroke\\b', '\\bcerebral infarction\\b', '\\bbrain infarction\\b', '\\blacunar stroke\\b', '\\bcardioembolic stroke\\b', '\\blarge artery stroke\\b', '\\bsmall vessel stroke\\b', '\\bsmall artery occlusion\\b', '\\btransient ischemic attack\\b', '\\btransient ischaemic attack\\b', '\\btia\\b']

def download_cytoband_if_needed(path_out):
    if os.path.exists(str(OUTPUT_ROOT / path_out if (OUTPUT_ROOT / path_out).exists() else INPUT_ROOT / path_out)):
        return
    url = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/cytoBand.txt.gz'
    print('Downloading cytoband file from:', url)
    urllib.request.urlretrieve(url, path_out)
    print('Saved:', path_out)

def parse_variant(v):
    m = re.match('(chr[0-9XY]+)_(\\d+)_', str(v))
    if m:
        return (m.group(1), int(m.group(2)))
    return (np.nan, np.nan)

def normalize_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()

def normalize_date_to_yyyy_mm_dd(x):
    dt = pd.to_datetime(x, errors='coerce')
    if pd.isna(dt):
        return normalize_text(x)
    return dt.strftime('%Y-%m-%d')

def band_prefix(band_name):
    m = re.match('^([pq]\\d+)', str(band_name))
    if m:
        return m.group(1)
    return str(band_name)

def is_coarse_band_name(x):
    return re.match('^[pq]\\d+$', str(x)) is not None

def build_regex(patterns):
    return re.compile('|'.join(patterns), flags=re.IGNORECASE)

def trait_match(series, regex_obj):
    return series.fillna('').astype(str).str.contains(regex_obj, regex=True, na=False)

def load_variant_positions(variant_csv):
    variant = pd.read_csv(str(OUTPUT_ROOT / variant_csv if (OUTPUT_ROOT / variant_csv).exists() else INPUT_ROOT / variant_csv))
    variant[['chr', 'pos']] = variant['variant_id'].apply(lambda x: pd.Series(parse_variant(x)))
    variant = variant.dropna(subset=['chr', 'pos']).copy()
    variant['pos'] = variant['pos'].astype(int)
    variant = variant[variant['chr'].isin(CHR_ORDER)].copy()
    return variant

def load_endpoint_ig(endpoint, variant_df, high_risk_q=0.85):
    ig_csv = IG_TEMPLATE.format(endpoint)
    pred_csv = PRED_TEMPLATE.format(endpoint)
    ig = pd.read_csv(str(OUTPUT_ROOT / ig_csv if (OUTPUT_ROOT / ig_csv).exists() else INPUT_ROOT / ig_csv))
    if 'ig_abs' in ig.columns:
        ig['abs_ig'] = ig['ig_abs'].abs()
    elif 'ig' in ig.columns:
        ig['abs_ig'] = ig['ig'].abs()
    else:
        raise ValueError(f"{ig_csv} must contain 'ig_abs' or 'ig'.")
    if high_risk_q is not None:
        pred = pd.read_csv(str(OUTPUT_ROOT / pred_csv if (OUTPUT_ROOT / pred_csv).exists() else INPUT_ROOT / pred_csv))
        pred['pid'] = pred['ID']
        risk_cut = pred['risk'].quantile(high_risk_q)
        high_pid = pred.loc[pred['risk'] >= risk_cut, 'pid']
        ig = ig[ig['pid'].isin(high_pid)].copy()
    ig = ig.merge(variant_df[['SNP', 'chr', 'pos']], left_on='feature', right_on='SNP', how='inner')
    snp_agg = ig.groupby(['SNP', 'chr', 'pos'], as_index=False)['abs_ig'].mean().rename(columns={'abs_ig': 'mean_abs_ig'})
    return snp_agg

def load_cytobands(path_in):
    cyto = pd.read_csv(str(OUTPUT_ROOT / path_in if (OUTPUT_ROOT / path_in).exists() else INPUT_ROOT / path_in), sep='\t', compression='gzip', header=None, names=['chrom', 'start', 'end', 'band_name', 'gieStain'])
    cyto = cyto[cyto['chrom'].isin(CHR_ORDER)].copy()
    cyto = cyto.sort_values(['chrom', 'start']).reset_index(drop=True)
    cyto['band_id'] = cyto['chrom'].astype(str) + ':' + cyto['band_name'].astype(str) + ':' + cyto['start'].astype(str) + '-' + cyto['end'].astype(str)
    cyto['start_mb'] = cyto['start'] / 1000000.0
    cyto['end_mb'] = cyto['end'] / 1000000.0
    cyto['len_bp'] = cyto['end'] - cyto['start']
    cyto['len_mb'] = cyto['len_bp'] / 1000000.0
    cyto['band_prefix'] = cyto['band_name'].apply(band_prefix)
    return cyto

def map_positions_to_bands(df, bands_df, chrom_col='chr', pos_col='pos'):
    if df.empty:
        return df.copy()
    out = []
    for (chrom, sub) in df.groupby(chrom_col):
        band_sub = bands_df[bands_df['chrom'] == chrom].sort_values('start').copy()
        if band_sub.empty:
            tmp = sub.copy()
            tmp['band_id'] = np.nan
            tmp['band_name'] = np.nan
            out.append(tmp)
            continue
        starts = band_sub['start'].to_numpy()
        ends = band_sub['end'].to_numpy()
        band_ids = band_sub['band_id'].to_numpy()
        band_names = band_sub['band_name'].to_numpy()
        pos = sub[pos_col].to_numpy()
        idx = np.searchsorted(starts, pos, side='right') - 1
        band_id_arr = np.array([np.nan] * len(sub), dtype=object)
        band_name_arr = np.array([np.nan] * len(sub), dtype=object)
        valid = idx >= 0
        valid_idx = idx[valid]
        valid_pos = pos[valid]
        inside = valid_pos < ends[valid_idx]
        pos_valid_indices = np.where(valid)[0][inside]
        band_valid_indices = valid_idx[inside]
        band_id_arr[pos_valid_indices] = band_ids[band_valid_indices]
        band_name_arr[pos_valid_indices] = band_names[band_valid_indices]
        tmp = sub.copy()
        tmp['band_id'] = band_id_arr
        tmp['band_name'] = band_name_arr
        out.append(tmp)
    out = pd.concat(out, ignore_index=True)
    return out

def compute_band_enrichment(band_table, disease_col_hits, total_col_hits, prefix):
    disease_total = band_table[disease_col_hits].sum()
    all_total = band_table[total_col_hits].sum()
    other_total = all_total - disease_total
    odds_list = []
    pval_list = []
    for (_, row) in band_table.iterrows():
        a = int(row[disease_col_hits])
        total_here = int(row[total_col_hits])
        b = max(total_here - a, 0)
        c = max(disease_total - a, 0)
        d = max(other_total - b, 0)
        table = np.array([[a, b], [c, d]], dtype=int)
        try:
            (odds_ratio, p_value) = fisher_exact(table, alternative='greater')
        except Exception:
            (odds_ratio, p_value) = (np.nan, np.nan)
        odds_list.append(odds_ratio)
        pval_list.append(p_value)
    band_table[f'{prefix}_odds_ratio'] = odds_list
    band_table[f'{prefix}_pvalue'] = pval_list
    return band_table

def decide_highlight(hit_count, or_value, p_value):
    if HIGHLIGHT_MODE == 'any_hit':
        return hit_count >= 1
    if HIGHLIGHT_MODE == 'or_and_p':
        if pd.isna(or_value) or pd.isna(p_value):
            return False
        return hit_count >= MIN_HITS and or_value >= MIN_OR and (p_value < MAX_P)
    return hit_count >= 1

def band_region_type(row):
    if row['chd_highlight'] and row['istroke_highlight']:
        return 'both'
    if row['chd_highlight']:
        return 'chd'
    if row['istroke_highlight']:
        return 'istroke'
    return 'none'

def chrom_sort_key(chrom):
    x = str(chrom).replace('chr', '')
    if x == 'X':
        return 23
    if x == 'Y':
        return 24
    return int(x)

def normalize_gwas_chr(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.upper() == 'X':
        return 'chrX'
    if s.upper() == 'Y':
        return 'chrY'
    if re.fullmatch('\\d+', s):
        return f'chr{int(s)}'
    if s == '23':
        return 'chrX'
    if s == '24':
        return 'chrY'
    return np.nan

def get_region_span(bands_df, chrom, start_band, end_band):
    sub_chr = bands_df[bands_df['chrom'] == chrom].sort_values('start').copy()
    if sub_chr.empty:
        print(f'[Warning] Cannot find chromosome: {chrom}')
        return None
    if is_coarse_band_name(start_band) and start_band == end_band:
        region_bands = sub_chr[sub_chr['band_prefix'] == start_band].copy()
        if region_bands.empty:
            print(f'[Warning] Cannot find coarse region: {chrom} {start_band}')
            return None
        return {'chrom': chrom, 'start_band': start_band, 'end_band': end_band, 'start_mb': region_bands['start_mb'].min(), 'end_mb': region_bands['end_mb'].max(), 'included_bands': ','.join(region_bands['band_name'].tolist())}
    start_row = sub_chr[sub_chr['band_name'] == start_band]
    end_row = sub_chr[sub_chr['band_name'] == end_band]
    if start_row.empty or end_row.empty:
        print(f'[Warning] Cannot find region: {chrom} {start_band}-{end_band}')
        return None
    left_bp = min(int(start_row['start'].iloc[0]), int(end_row['start'].iloc[0]))
    right_bp = max(int(start_row['end'].iloc[0]), int(end_row['end'].iloc[0]))
    region_bands = sub_chr[(sub_chr['end'] > left_bp) & (sub_chr['start'] < right_bp)].copy()
    if region_bands.empty:
        return None
    return {'chrom': chrom, 'start_band': start_band, 'end_band': end_band, 'start_mb': region_bands['start_mb'].min(), 'end_mb': region_bands['end_mb'].max(), 'included_bands': ','.join(region_bands['band_name'].tolist())}

def apply_manual_study_flags_all_hits(gwas_band, manual_studies):
    gwas_band = gwas_band.copy()
    gwas_band['is_chd_related'] = False
    gwas_band['is_istroke_related'] = False
    gwas_band['manual_study_selected'] = False
    for col in ['FIRST AUTHOR', 'JOURNAL', 'PUBMEDID']:
        if col in gwas_band.columns:
            gwas_band[f'__{col}'] = gwas_band[col].apply(normalize_text)
        else:
            gwas_band[f'__{col}'] = ''
    gwas_band['__DATE'] = gwas_band['DATE'].apply(normalize_date_to_yyyy_mm_dd)
    trait_series = gwas_band['DISEASE/TRAIT'].fillna('').astype(str)
    study_series = gwas_band.get('STUDY', pd.Series([''] * len(gwas_band), index=gwas_band.index)).fillna('').astype(str)
    chd_regex = build_regex(CHD_PATTERNS)
    istroke_regex = build_regex(ISTROKE_PATTERNS)
    chd_trait_mask = trait_match(trait_series, chd_regex) | trait_match(study_series, chd_regex)
    istroke_trait_mask = trait_match(trait_series, istroke_regex) | trait_match(study_series, istroke_regex)
    for item in manual_studies:
        endpoint = item['endpoint']
        author = normalize_text(item['FIRST AUTHOR'])
        date = normalize_date_to_yyyy_mm_dd(item['DATE'])
        journal = normalize_text(item['JOURNAL'])
        pubmedid = normalize_text(item['PUBMEDID'])
        study_mask = (gwas_band['__FIRST AUTHOR'] == author) & (gwas_band['__DATE'] == date) & (gwas_band['__JOURNAL'] == journal) & (gwas_band['__PUBMEDID'] == pubmedid)
        if endpoint == 'CHD':
            final_mask = study_mask & chd_trait_mask
            gwas_band.loc[final_mask, 'is_chd_related'] = True
        elif endpoint == 'istroke':
            final_mask = study_mask & istroke_trait_mask
            gwas_band.loc[final_mask, 'is_istroke_related'] = True
        else:
            raise ValueError(f'Unknown endpoint in MANUAL_SELECTED_STUDIES: {endpoint}')
        gwas_band.loc[study_mask, 'manual_study_selected'] = True
        print(f'[Manual study] {endpoint} | {author} {date} {journal} PMID={pubmedid} | all hits in study = {int(study_mask.sum())} | endpoint-matched hits = {int(final_mask.sum())}')
    helper_cols = [c for c in gwas_band.columns if c.startswith('__')]
    gwas_band = gwas_band.drop(columns=helper_cols)
    return gwas_band
download_cytoband_if_needed(CYTOBAND_FILE)
bands = load_cytobands(CYTOBAND_FILE)
variant = load_variant_positions(VARIANT_CSV)
chrom_sizes = bands.groupby('chrom', as_index=False)['end_mb'].max().rename(columns={'end_mb': 'chrom_len_mb'})
chrom_len_dict = dict(zip(chrom_sizes['chrom'], chrom_sizes['chrom_len_mb']))
snp_chd = load_endpoint_ig('CHD', variant, high_risk_q=HIGH_RISK_Q)
snp_istroke = load_endpoint_ig('istroke', variant, high_risk_q=HIGH_RISK_Q)
snp_chd_band = map_positions_to_bands(snp_chd, bands, chrom_col='chr', pos_col='pos')
snp_istroke_band = map_positions_to_bands(snp_istroke, bands, chrom_col='chr', pos_col='pos')
gwas = pd.read_csv(str(OUTPUT_ROOT / GWAS_CATALOG if (OUTPUT_ROOT / GWAS_CATALOG).exists() else INPUT_ROOT / GWAS_CATALOG), sep='\t', low_memory=False)
gwas.columns = gwas.columns.str.strip()
gwas['CHR_POS'] = pd.to_numeric(gwas['CHR_POS'], errors='coerce')
gwas = gwas.dropna(subset=['CHR_ID', 'CHR_POS']).copy()
gwas['chr'] = gwas['CHR_ID'].apply(normalize_gwas_chr)
gwas = gwas[gwas['chr'].isin(CHR_ORDER)].copy()
gwas['pos'] = gwas['CHR_POS'].astype(int)
print('\nGWAS chromosome counts after normalization:')
print(gwas['chr'].value_counts().sort_index())
gwas_band = map_positions_to_bands(gwas, bands, chrom_col='chr', pos_col='pos')
gwas_band = gwas_band.dropna(subset=['band_id']).copy()
print('\nGWAS mapped cytoband chromosome counts:')
print(gwas_band['chr'].value_counts().sort_index())
gwas_band = apply_manual_study_flags_all_hits(gwas_band=gwas_band, manual_studies=MANUAL_SELECTED_STUDIES)
manual_selected_hits = gwas_band[gwas_band['is_chd_related'] | gwas_band['is_istroke_related']].copy()
manual_selected_hits.to_csv(OUT_SELECTED_STUDIES, index=False)
print('\nSaved manual selected GWAS hits:')
print(OUT_SELECTED_STUDIES)
print('\nManual selected GWAS hits on chrX:')
if (manual_selected_hits['chr'] == 'chrX').any():
    print(manual_selected_hits.loc[manual_selected_hits['chr'] == 'chrX', [c for c in ['chr', 'pos', 'band_name', 'DISEASE/TRAIT', 'STUDY', 'FIRST AUTHOR', 'DATE', 'JOURNAL', 'PUBMEDID'] if c in manual_selected_hits.columns]].head(50).to_string(index=False))
else:
    print('No chrX hits matched the current MANUAL_SELECTED_STUDIES + disease keyword criteria.')
if not manual_selected_hits.empty:
    summary_group_cols = [c for c in ['is_chd_related', 'is_istroke_related', 'DISEASE/TRAIT', 'STUDY', 'FIRST AUTHOR', 'DATE', 'JOURNAL', 'PUBMEDID'] if c in manual_selected_hits.columns]
    manual_study_summary = manual_selected_hits.groupby(summary_group_cols, dropna=False).agg(n_gwas_hits=('SNPS', 'count') if 'SNPS' in manual_selected_hits.columns else ('DISEASE/TRAIT', 'count'), min_pvalue=('P-VALUE', 'min') if 'P-VALUE' in manual_selected_hits.columns else ('DISEASE/TRAIT', 'count')).reset_index()
    manual_study_summary.to_csv(OUT_SELECTED_STUDY_SUMMARY, index=False)
    print('\nSaved manual selected study summary:')
    print(OUT_SELECTED_STUDY_SUMMARY)
band_table = bands[['band_id', 'chrom', 'band_name', 'start', 'end', 'start_mb', 'end_mb', 'len_bp', 'len_mb']].copy()
all_counts = gwas_band.groupby('band_id', as_index=False).size().rename(columns={'size': 'all_gwas_hits'})
chd_counts = gwas_band[gwas_band['is_chd_related']].groupby('band_id', as_index=False).size().rename(columns={'size': 'chd_gwas_hits'})
istroke_counts = gwas_band[gwas_band['is_istroke_related']].groupby('band_id', as_index=False).size().rename(columns={'size': 'istroke_gwas_hits'})
band_table = band_table.merge(all_counts, on='band_id', how='left')
band_table = band_table.merge(chd_counts, on='band_id', how='left')
band_table = band_table.merge(istroke_counts, on='band_id', how='left')
for col in ['all_gwas_hits', 'chd_gwas_hits', 'istroke_gwas_hits']:
    band_table[col] = band_table[col].fillna(0).astype(int)
band_table = compute_band_enrichment(band_table=band_table, disease_col_hits='chd_gwas_hits', total_col_hits='all_gwas_hits', prefix='chd')
band_table = compute_band_enrichment(band_table=band_table, disease_col_hits='istroke_gwas_hits', total_col_hits='all_gwas_hits', prefix='istroke')
chd_ig_band = snp_chd_band.dropna(subset=['band_id']).groupby('band_id', as_index=False)['mean_abs_ig'].mean().rename(columns={'mean_abs_ig': 'chd_mean_abs_ig'})
istroke_ig_band = snp_istroke_band.dropna(subset=['band_id']).groupby('band_id', as_index=False)['mean_abs_ig'].mean().rename(columns={'mean_abs_ig': 'istroke_mean_abs_ig'})
band_table = band_table.merge(chd_ig_band, on='band_id', how='left')
band_table = band_table.merge(istroke_ig_band, on='band_id', how='left')
band_table['chd_highlight'] = band_table.apply(lambda r: decide_highlight(hit_count=r['chd_gwas_hits'], or_value=r['chd_odds_ratio'], p_value=r['chd_pvalue']), axis=1)
band_table['istroke_highlight'] = band_table.apply(lambda r: decide_highlight(hit_count=r['istroke_gwas_hits'], or_value=r['istroke_odds_ratio'], p_value=r['istroke_pvalue']), axis=1)
band_table['region_type'] = band_table.apply(band_region_type, axis=1)
band_table = band_table.sort_values(['chrom', 'start']).reset_index(drop=True)
band_table.to_csv(OUT_BAND_TABLE, index=False)
snp_band_summary = snp_chd_band[['SNP', 'chr', 'pos', 'mean_abs_ig', 'band_id']].rename(columns={'mean_abs_ig': 'chd_mean_abs_ig'}).merge(snp_istroke_band[['SNP', 'chr', 'pos', 'mean_abs_ig', 'band_id']].rename(columns={'mean_abs_ig': 'istroke_mean_abs_ig'}), on=['SNP', 'chr', 'pos', 'band_id'], how='outer')
snp_band_summary.to_csv(OUT_SNP_TABLE, index=False)
print('Saved band summary with OR:', OUT_BAND_TABLE)
print('Saved SNP-band IG table:', OUT_SNP_TABLE)
chd_q75 = band_table['chd_mean_abs_ig'].dropna().quantile(PLOT_IG_Q)
istroke_q75 = band_table['istroke_mean_abs_ig'].dropna().quantile(PLOT_IG_Q)
band_table['chd_q75_region'] = band_table['chd_mean_abs_ig'].notna() & (band_table['chd_mean_abs_ig'] >= chd_q75)
band_table['istroke_q75_region'] = band_table['istroke_mean_abs_ig'].notna() & (band_table['istroke_mean_abs_ig'] >= istroke_q75)
band_table['any_q75_region'] = band_table['chd_q75_region'] | band_table['istroke_q75_region']
q75_table = band_table[band_table['any_q75_region']].copy()
q75_table.to_csv(OUT_Q75_TABLE, index=False)
if PLOT_ALL_CHROMOSOMES:
    plot_chroms = [c for c in CHR_ORDER if c in chrom_len_dict]
else:
    label_chroms = [x[0] for x in LABEL_REGIONS]
    plot_chroms = q75_table['chrom'].drop_duplicates().tolist()
    plot_chroms = sorted(set(plot_chroms) | set(label_chroms), key=chrom_sort_key)
print('\nCHD q75 cutoff:', chd_q75)
print('istroke q75 cutoff:', istroke_q75)
print('Chromosomes kept for plotting:', plot_chroms)
print('Saved q75 selected bands:', OUT_Q75_TABLE)
ig_values_all = pd.concat([band_table['chd_mean_abs_ig'], band_table['istroke_mean_abs_ig']], axis=0).dropna()
if len(ig_values_all) > 0:
    vmin = ig_values_all.quantile(0.01)
    vmax = ig_values_all.quantile(0.99)
    if vmin == vmax:
        vmin = ig_values_all.min()
        vmax = ig_values_all.max()
else:
    (vmin, vmax) = (0.0, 1.0)
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = HEATMAP_CMAP
label_region_spans = []
for (chrom, start_band, end_band, label) in LABEL_REGIONS:
    span = get_region_span(bands, chrom, start_band, end_band)
    if span is not None:
        span['label'] = label
        label_region_spans.append(span)
fig_height = max(5.0, 0.75 * len(plot_chroms) + 2.0)
(fig, ax) = plt.subplots(figsize=(FIGSIZE[0], fig_height))
y_positions = {chrom: i * Y_GAP for (i, chrom) in enumerate(plot_chroms)}
max_chr_len = max([chrom_len_dict[c] for c in plot_chroms])
label_offset_counter = {chrom: 0 for chrom in plot_chroms}
for chrom in plot_chroms:
    if chrom not in chrom_len_dict:
        continue
    y0 = y_positions[chrom]
    sub = band_table[band_table['chrom'] == chrom].sort_values('start').copy()
    if sub.empty:
        continue
    for (_, row) in sub.iterrows():
        x = row['start_mb']
        w = row['len_mb']
        if row['region_type'] == 'chd':
            face = COLOR_CHD
        elif row['region_type'] == 'istroke':
            face = COLOR_ISTROKE
        elif row['region_type'] == 'both':
            face = COLOR_BOTH
        else:
            face = COLOR_NONE
        ax.add_patch(Rectangle((x, y0 - CHROM_HEIGHT / 2), w, CHROM_HEIGHT, facecolor=face, edgecolor='white', linewidth=0.15, zorder=2))
    ax.add_patch(Rectangle((0, y0 - CHROM_HEIGHT / 2), chrom_len_dict[chrom], CHROM_HEIGHT, facecolor='none', edgecolor=EDGE_COLOR, linewidth=0.8, zorder=3))
    chd_track_y = y0 + CHROM_HEIGHT / 2 + CHROM_TO_TRACK_GAP
    istroke_track_y = chd_track_y + TRACK_HEIGHT + TRACK_GAP
    for (_, row) in sub.iterrows():
        x = row['start_mb']
        w = row['len_mb']
        if row['chd_q75_region'] and pd.notna(row['chd_mean_abs_ig']):
            color_chd = cmap(norm(row['chd_mean_abs_ig']))
        else:
            color_chd = '#FFFFFF'
        if row['istroke_q75_region'] and pd.notna(row['istroke_mean_abs_ig']):
            color_ist = cmap(norm(row['istroke_mean_abs_ig']))
        else:
            color_ist = '#FFFFFF'
        ax.add_patch(Rectangle((x, chd_track_y), w, TRACK_HEIGHT, facecolor=color_chd, edgecolor='white', linewidth=0.12, zorder=2))
        ax.add_patch(Rectangle((x, istroke_track_y), w, TRACK_HEIGHT, facecolor=color_ist, edgecolor='white', linewidth=0.12, zorder=2))
    ax.add_patch(Rectangle((0, chd_track_y), chrom_len_dict[chrom], TRACK_HEIGHT, facecolor='none', edgecolor=EDGE_COLOR, linewidth=0.6, zorder=3))
    ax.add_patch(Rectangle((0, istroke_track_y), chrom_len_dict[chrom], TRACK_HEIGHT, facecolor='none', edgecolor=EDGE_COLOR, linewidth=0.6, zorder=3))
    ax.text(-6, y0, chrom, ha='right', va='center', fontsize=9)
    ax.text(-6, chd_track_y + TRACK_HEIGHT / 2, 'CHD', ha='right', va='center', fontsize=8)
    ax.text(-6, istroke_track_y + TRACK_HEIGHT / 2, 'istroke', ha='right', va='center', fontsize=8)
    spans_this_chrom = [s for s in label_region_spans if s['chrom'] == chrom]
    for span in spans_this_chrom:
        x = span['start_mb']
        w = span['end_mb'] - span['start_mb']
        box_y = y0 - CHROM_HEIGHT / 2 - 0.01
        box_h = istroke_track_y + TRACK_HEIGHT - box_y + 0.01
        ax.add_patch(Rectangle((x, box_y), w, box_h, facecolor='none', edgecolor=LABEL_EDGE_COLOR, linewidth=LABEL_LINEWIDTH, zorder=6))
        k = label_offset_counter[chrom]
        label_offset_counter[chrom] += 1
        text_x = x + w / 2
        text_y = y0 - CHROM_HEIGHT / 2 - 0.08 - 0.11 * (k % 3)
        ax.plot([x + w / 2, x + w / 2], [box_y, text_y + 0.02], color=LABEL_EDGE_COLOR, linewidth=0.9, zorder=6)
        ax.text(text_x, text_y, span['label'], color=LABEL_TEXT_COLOR, fontsize=LABEL_FONTSIZE, ha='center', va='bottom', zorder=7)
legend_handles = [Patch(facecolor=COLOR_CHD, edgecolor='none', label='Selected CHD study region'), Patch(facecolor=COLOR_ISTROKE, edgecolor='none', label='Selected istroke study region'), Patch(facecolor=COLOR_BOTH, edgecolor='none', label='Shared selected study region'), Patch(facecolor='#FFFFFF', edgecolor=EDGE_COLOR, label='No q75 IG signal'), Patch(facecolor='white', edgecolor=LABEL_EDGE_COLOR, label='Highlighted labeled region')]
ax.legend(handles=legend_handles, frameon=False, fontsize=8, title='Regions', title_fontsize=9, loc='upper left', bbox_to_anchor=(1.02, 0.98))
ax.set_xlim(-12, max_chr_len + 8)
y_min = -0.8
y_max = max(y_positions.values()) + CHROM_HEIGHT + CHROM_TO_TRACK_GAP + 2 * TRACK_HEIGHT + TRACK_GAP + 0.75
ax.set_ylim(y_max, y_min)
ax.set_xlabel('Genomic position (Mb)')
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
plt.tight_layout()
plt.savefig(OUTFIG, dpi=300, transparent=True, bbox_inches='tight')
plt.close()
print('Saved final figure:', OUTFIG)
(fig_cb, ax_cb) = plt.subplots(figsize=(1.2, 3.2))
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, cax=ax_cb, orientation='vertical')
cb_ticks = np.linspace(vmin, vmax, 3)
cbar.set_ticks(cb_ticks)
cbar.set_ticklabels([f'{x:.4f}' for x in cb_ticks])
cbar.set_label('Mean absolute IG', fontsize=10)
cbar.ax.tick_params(labelsize=8)
plt.savefig(OUT_COLORBAR, dpi=300, transparent=True, bbox_inches='tight')
plt.close(fig_cb)
print('Saved separate colorbar:', OUT_COLORBAR)
print('\nQ75 selected regions:')
print(q75_table[['chrom', 'band_name', 'chd_mean_abs_ig', 'istroke_mean_abs_ig', 'chd_q75_region', 'istroke_q75_region', 'region_type']].to_string(index=False))
print('\nchrX rows in band_table:')
print(band_table.loc[band_table['chrom'] == 'chrX', ['chrom', 'band_name', 'all_gwas_hits', 'chd_gwas_hits', 'istroke_gwas_hits', 'chd_mean_abs_ig', 'istroke_mean_abs_ig', 'region_type']].to_string(index=False))
print('\nLabeled regions in final plot:')
for s in label_region_spans:
    print(f"{s['chrom']} {s['start_band']}-{s['end_band']} ({s['start_mb']:.2f}-{s['end_mb']:.2f} Mb): {s['label']} [included bands: {s['included_bands']}]")


### Extended Data Figure 3b — Chromosome coverage


In [ ]:
print('Cytoband chromosomes:')
print(sorted(bands['chrom'].unique(), key=chrom_sort_key))
print('\nVariant chromosomes:')
print(variant['chr'].value_counts().sort_index())
print('\nRaw GWAS CHR_ID examples:')
print(pd.read_csv(str(OUTPUT_ROOT / GWAS_CATALOG if (OUTPUT_ROOT / GWAS_CATALOG).exists() else INPUT_ROOT / GWAS_CATALOG), sep='\t', low_memory=False)['CHR_ID'].astype(str).value_counts().head(30))


### Extended Data Figure 3b — Genome-wide attribution, two-row layout


In [ ]:
import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from scipy.stats import fisher_exact
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
ENDPOINTS = ['CHD', 'istroke']
HIGH_RISK_Q = 0.85
IG_TEMPLATE = 'code_02_02_save/snp_20260202_1432/explanations/{}_snp_TR_best_IG.csv'
PRED_TEMPLATE = 'code_02_02_save/snp_20260202_1432/preds/{}_snp_TR_test_best_pred.csv'
VARIANT_CSV = 'CHD_SNP_variant_gene_with_symbol.csv'
GWAS_CATALOG = 'gwas-catalog-download-associations-v1.0-full.tsv'
OUTDIR = 'figs3_final/chromosome_IG_vertical_2rows_all_chromosomes'
os.makedirs(OUTDIR, exist_ok=True)
CYTOBAND_FILE = os.path.join(OUTDIR, 'hg38_cytoBand.txt.gz')
OUTFIG = os.path.join(OUTDIR, 'CHD_istroke_all_chromosomes_vertical_2rows_q75_heatmap.pdf')
OUTSVG = os.path.join(OUTDIR, 'CHD_istroke_all_chromosomes_vertical_2rows_q75_heatmap.svg')
OUT_COLORBAR = os.path.join(OUTDIR, 'CHD_istroke_all_chromosomes_vertical_2rows_q75_heatmap_colorbar.pdf')
OUT_BAND_TABLE = os.path.join(OUTDIR, 'CHD_istroke_cytoband_summary_with_OR_manual_studies_all_hits_chrX_public.csv')
OUT_SNP_TABLE = os.path.join(OUTDIR, 'CHD_istroke_snp_band_ig_summary_manual_studies_all_hits_chrX_public.csv')
OUT_Q75_TABLE = os.path.join(OUTDIR, 'CHD_istroke_q75_selected_bands_manual_studies_all_hits_chrX_public.csv')
OUT_SELECTED_STUDIES = os.path.join(OUTDIR, 'manual_selected_CHD_istroke_GWAS_hits_all_hits_chrX_public.csv')
OUT_SELECTED_STUDY_SUMMARY = os.path.join(OUTDIR, 'manual_selected_CHD_istroke_GWAS_study_summary_all_hits_chrX_public.csv')
CHR_ORDER = [f'chr{i}' for i in range(1, 23)] + ['chrX']
MANUAL_SELECTED_STUDIES = [{'endpoint': 'istroke', 'FIRST AUTHOR': 'Malik R', 'DATE': '2018-03-12', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '29531354'}, {'endpoint': 'istroke', 'FIRST AUTHOR': 'Mishra A', 'DATE': '2022-09-30', 'JOURNAL': 'Nature', 'PUBMEDID': '36180795'}, {'endpoint': 'istroke', 'FIRST AUTHOR': 'Soderholm M', 'DATE': '2019-02-22', 'JOURNAL': 'Neurology', 'PUBMEDID': '30796134'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Aragam KG', 'DATE': '2022-12-06', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '36474045'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Tcheandjieu C', 'DATE': '2022-08-01', 'JOURNAL': 'Nat Med', 'PUBMEDID': '35915156'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Koyama S', 'DATE': '2020-10-05', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '33020668'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Lu X', 'DATE': '2012-07-01', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '22751097'}, {'endpoint': 'CHD', 'FIRST AUTHOR': 'Nikpay M', 'DATE': '2015-09-07', 'JOURNAL': 'Nat Genet', 'PUBMEDID': '26343387'}]
LABEL_REGIONS = [('chr9', 'p21', 'p21', '9p21 / CDKN2A-CDKN2B-ANRIL'), ('chr7', 'p21', 'p21', '7p21 / HDAC9'), ('chr6', 'p24.1', 'p24.1', '6p24.1 / PHACTR1'), ('chr6', 'q26', 'q26', '6q26 / LPA'), ('chr19', 'p13', 'p13', '19p13 / LDLR'), ('chr1', 'p13', 'p13', '1p13 / SORT1'), ('chr1', 'p32', 'p32', '1p32 / PCSK9'), ('chr9', 'q33.2', 'q33.2', '9q33.2'), ('chr5', 'q22', 'q22', '5q22'), ('chr10', 'q23.31', 'q23.31', '10q23.31 / LIPA'), ('chr16', 'q24.2', 'q24.2', '16q24.2')]
PLOT_ALL_CHROMOSOMES = True
FIGSIZE_MM = (200, 200)
FIGSIZE = (FIGSIZE_MM[0] / 25.4, FIGSIZE_MM[1] / 25.4)
CHROM_WIDTH = 0.28
TRACK_WIDTH = 0.095
TRACK_GAP = 0.055
CHROM_TO_TRACK_GAP = 0.08
COL_GAP = 0.75
ROW_GAP_MB = 55
TOP_LABEL_PAD_MB = 10
BOTTOM_PAD_MB = 20
HEATMAP_CMAP = LinearSegmentedColormap.from_list('ig_grey_black', ['#EEEEEE', '#111111'])
COLOR_NONE = '#FFFFFF'
COLOR_CHD = '#D55E5E'
COLOR_ISTROKE = '#5B8FD9'
COLOR_BOTH = '#9B59B6'
EDGE_COLOR = '#A8A8A8'
LABEL_EDGE_COLOR = '#D62728'
LABEL_TEXT_COLOR = '#B22222'
LABEL_LINEWIDTH = 1.1
LABEL_FONTSIZE = 5.5
PLOT_IG_Q = 0.75
HIGHLIGHT_MODE = 'any_hit'
MIN_OR = 1.0
MAX_P = 0.05
MIN_HITS = 1
CHD_PATTERNS = ['\\bcoronary heart disease\\b', '\\bcoronary artery disease\\b', '\\bischemic heart disease\\b', '\\bmyocardial infarction\\b', '\\bangina\\b', '\\bunstable angina\\b', '\\bacute coronary syndrome\\b', '\\batherosclerosis\\b', '\\bcoronary atherosclerosis\\b', '\\bcoronary stenosis\\b', '\\bchd\\b', '\\bcad\\b']
ISTROKE_PATTERNS = ['\\bischemic stroke\\b', '\\bischaemic stroke\\b', '\\bstroke\\b', '\\bcerebral infarction\\b', '\\bbrain infarction\\b', '\\blacunar stroke\\b', '\\bcardioembolic stroke\\b', '\\blarge artery stroke\\b', '\\bsmall vessel stroke\\b', '\\bsmall artery occlusion\\b', '\\btransient ischemic attack\\b', '\\btransient ischaemic attack\\b', '\\btia\\b']

def download_cytoband_if_needed(path_out):
    if os.path.exists(str(OUTPUT_ROOT / path_out if (OUTPUT_ROOT / path_out).exists() else INPUT_ROOT / path_out)):
        return
    url = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/cytoBand.txt.gz'
    print('Downloading cytoband file from:', url)
    urllib.request.urlretrieve(url, path_out)
    print('Saved:', path_out)

def parse_variant(v):
    m = re.match('(chr[0-9XY]+)_(\\d+)_', str(v))
    if m:
        return (m.group(1), int(m.group(2)))
    return (np.nan, np.nan)

def normalize_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()

def normalize_date_to_yyyy_mm_dd(x):
    dt = pd.to_datetime(x, errors='coerce')
    if pd.isna(dt):
        return normalize_text(x)
    return dt.strftime('%Y-%m-%d')

def band_prefix(band_name):
    m = re.match('^([pq]\\d+)', str(band_name))
    if m:
        return m.group(1)
    return str(band_name)

def is_coarse_band_name(x):
    return re.match('^[pq]\\d+$', str(x)) is not None

def build_regex(patterns):
    return re.compile('|'.join(patterns), flags=re.IGNORECASE)

def trait_match(series, regex_obj):
    return series.fillna('').astype(str).str.contains(regex_obj, regex=True, na=False)

def chrom_sort_key(chrom):
    x = str(chrom).replace('chr', '')
    if x == 'X':
        return 23
    if x == 'Y':
        return 24
    return int(x)

def normalize_gwas_chr(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.upper() == 'X':
        return 'chrX'
    if s.upper() == 'Y':
        return 'chrY'
    if re.fullmatch('\\d+', s):
        val = int(s)
        if val == 23:
            return 'chrX'
        if val == 24:
            return 'chrY'
        return f'chr{val}'
    return np.nan

def load_variant_positions(variant_csv):
    variant = pd.read_csv(str(OUTPUT_ROOT / variant_csv if (OUTPUT_ROOT / variant_csv).exists() else INPUT_ROOT / variant_csv))
    variant[['chr', 'pos']] = variant['variant_id'].apply(lambda x: pd.Series(parse_variant(x)))
    variant = variant.dropna(subset=['chr', 'pos']).copy()
    variant['pos'] = variant['pos'].astype(int)
    variant = variant[variant['chr'].isin(CHR_ORDER)].copy()
    return variant

def load_endpoint_ig(endpoint, variant_df, high_risk_q=0.85):
    ig_csv = IG_TEMPLATE.format(endpoint)
    pred_csv = PRED_TEMPLATE.format(endpoint)
    ig = pd.read_csv(str(OUTPUT_ROOT / ig_csv if (OUTPUT_ROOT / ig_csv).exists() else INPUT_ROOT / ig_csv))
    if 'ig_abs' in ig.columns:
        ig['abs_ig'] = ig['ig_abs'].abs()
    elif 'ig' in ig.columns:
        ig['abs_ig'] = ig['ig'].abs()
    else:
        raise ValueError(f"{ig_csv} must contain 'ig_abs' or 'ig'.")
    if high_risk_q is not None:
        pred = pd.read_csv(str(OUTPUT_ROOT / pred_csv if (OUTPUT_ROOT / pred_csv).exists() else INPUT_ROOT / pred_csv))
        pred['pid'] = pred['ID']
        risk_cut = pred['risk'].quantile(high_risk_q)
        high_pid = pred.loc[pred['risk'] >= risk_cut, 'pid']
        ig = ig[ig['pid'].isin(high_pid)].copy()
    ig = ig.merge(variant_df[['SNP', 'chr', 'pos']], left_on='feature', right_on='SNP', how='inner')
    snp_agg = ig.groupby(['SNP', 'chr', 'pos'], as_index=False)['abs_ig'].mean().rename(columns={'abs_ig': 'mean_abs_ig'})
    return snp_agg

def load_cytobands(path_in):
    cyto = pd.read_csv(str(OUTPUT_ROOT / path_in if (OUTPUT_ROOT / path_in).exists() else INPUT_ROOT / path_in), sep='\t', compression='gzip', header=None, names=['chrom', 'start', 'end', 'band_name', 'gieStain'])
    cyto = cyto[cyto['chrom'].isin(CHR_ORDER)].copy()
    cyto = cyto.sort_values(['chrom', 'start']).reset_index(drop=True)
    cyto['band_id'] = cyto['chrom'].astype(str) + ':' + cyto['band_name'].astype(str) + ':' + cyto['start'].astype(str) + '-' + cyto['end'].astype(str)
    cyto['start_mb'] = cyto['start'] / 1000000.0
    cyto['end_mb'] = cyto['end'] / 1000000.0
    cyto['len_bp'] = cyto['end'] - cyto['start']
    cyto['len_mb'] = cyto['len_bp'] / 1000000.0
    cyto['band_prefix'] = cyto['band_name'].apply(band_prefix)
    return cyto

def map_positions_to_bands(df, bands_df, chrom_col='chr', pos_col='pos'):
    if df.empty:
        return df.copy()
    out = []
    for (chrom, sub) in df.groupby(chrom_col):
        band_sub = bands_df[bands_df['chrom'] == chrom].sort_values('start').copy()
        if band_sub.empty:
            tmp = sub.copy()
            tmp['band_id'] = np.nan
            tmp['band_name'] = np.nan
            out.append(tmp)
            continue
        starts = band_sub['start'].to_numpy()
        ends = band_sub['end'].to_numpy()
        band_ids = band_sub['band_id'].to_numpy()
        band_names = band_sub['band_name'].to_numpy()
        pos = sub[pos_col].to_numpy()
        idx = np.searchsorted(starts, pos, side='right') - 1
        band_id_arr = np.array([np.nan] * len(sub), dtype=object)
        band_name_arr = np.array([np.nan] * len(sub), dtype=object)
        valid = idx >= 0
        valid_idx = idx[valid]
        valid_pos = pos[valid]
        inside = valid_pos < ends[valid_idx]
        pos_valid_indices = np.where(valid)[0][inside]
        band_valid_indices = valid_idx[inside]
        band_id_arr[pos_valid_indices] = band_ids[band_valid_indices]
        band_name_arr[pos_valid_indices] = band_names[band_valid_indices]
        tmp = sub.copy()
        tmp['band_id'] = band_id_arr
        tmp['band_name'] = band_name_arr
        out.append(tmp)
    return pd.concat(out, ignore_index=True)

def compute_band_enrichment(band_table, disease_col_hits, total_col_hits, prefix):
    disease_total = band_table[disease_col_hits].sum()
    all_total = band_table[total_col_hits].sum()
    other_total = all_total - disease_total
    odds_list = []
    pval_list = []
    for (_, row) in band_table.iterrows():
        a = int(row[disease_col_hits])
        total_here = int(row[total_col_hits])
        b = max(total_here - a, 0)
        c = max(disease_total - a, 0)
        d = max(other_total - b, 0)
        table = np.array([[a, b], [c, d]], dtype=int)
        try:
            (odds_ratio, p_value) = fisher_exact(table, alternative='greater')
        except Exception:
            (odds_ratio, p_value) = (np.nan, np.nan)
        odds_list.append(odds_ratio)
        pval_list.append(p_value)
    band_table[f'{prefix}_odds_ratio'] = odds_list
    band_table[f'{prefix}_pvalue'] = pval_list
    return band_table

def decide_highlight(hit_count, or_value, p_value):
    if HIGHLIGHT_MODE == 'any_hit':
        return hit_count >= 1
    if HIGHLIGHT_MODE == 'or_and_p':
        if pd.isna(or_value) or pd.isna(p_value):
            return False
        return hit_count >= MIN_HITS and or_value >= MIN_OR and (p_value < MAX_P)
    return hit_count >= 1

def band_region_type(row):
    if row['chd_highlight'] and row['istroke_highlight']:
        return 'both'
    if row['chd_highlight']:
        return 'chd'
    if row['istroke_highlight']:
        return 'istroke'
    return 'none'

def get_region_span(bands_df, chrom, start_band, end_band):
    sub_chr = bands_df[bands_df['chrom'] == chrom].sort_values('start').copy()
    if sub_chr.empty:
        print(f'[Warning] Cannot find chromosome: {chrom}')
        return None
    if is_coarse_band_name(start_band) and start_band == end_band:
        region_bands = sub_chr[sub_chr['band_prefix'] == start_band].copy()
        if region_bands.empty:
            print(f'[Warning] Cannot find coarse region: {chrom} {start_band}')
            return None
        return {'chrom': chrom, 'start_band': start_band, 'end_band': end_band, 'start_mb': region_bands['start_mb'].min(), 'end_mb': region_bands['end_mb'].max(), 'included_bands': ','.join(region_bands['band_name'].tolist())}
    start_row = sub_chr[sub_chr['band_name'] == start_band]
    end_row = sub_chr[sub_chr['band_name'] == end_band]
    if start_row.empty or end_row.empty:
        print(f'[Warning] Cannot find region: {chrom} {start_band}-{end_band}')
        return None
    left_bp = min(int(start_row['start'].iloc[0]), int(end_row['start'].iloc[0]))
    right_bp = max(int(start_row['end'].iloc[0]), int(end_row['end'].iloc[0]))
    region_bands = sub_chr[(sub_chr['end'] > left_bp) & (sub_chr['start'] < right_bp)].copy()
    if region_bands.empty:
        return None
    return {'chrom': chrom, 'start_band': start_band, 'end_band': end_band, 'start_mb': region_bands['start_mb'].min(), 'end_mb': region_bands['end_mb'].max(), 'included_bands': ','.join(region_bands['band_name'].tolist())}

def apply_manual_study_flags_all_hits(gwas_band, manual_studies):
    gwas_band = gwas_band.copy()
    gwas_band['is_chd_related'] = False
    gwas_band['is_istroke_related'] = False
    gwas_band['manual_study_selected'] = False
    for col in ['FIRST AUTHOR', 'JOURNAL', 'PUBMEDID']:
        if col in gwas_band.columns:
            gwas_band[f'__{col}'] = gwas_band[col].apply(normalize_text)
        else:
            gwas_band[f'__{col}'] = ''
    gwas_band['__DATE'] = gwas_band['DATE'].apply(normalize_date_to_yyyy_mm_dd)
    trait_series = gwas_band['DISEASE/TRAIT'].fillna('').astype(str)
    study_series = gwas_band.get('STUDY', pd.Series([''] * len(gwas_band), index=gwas_band.index)).fillna('').astype(str)
    chd_regex = build_regex(CHD_PATTERNS)
    istroke_regex = build_regex(ISTROKE_PATTERNS)
    chd_trait_mask = trait_match(trait_series, chd_regex) | trait_match(study_series, chd_regex)
    istroke_trait_mask = trait_match(trait_series, istroke_regex) | trait_match(study_series, istroke_regex)
    for item in manual_studies:
        endpoint = item['endpoint']
        author = normalize_text(item['FIRST AUTHOR'])
        date = normalize_date_to_yyyy_mm_dd(item['DATE'])
        journal = normalize_text(item['JOURNAL'])
        pubmedid = normalize_text(item['PUBMEDID'])
        study_mask = (gwas_band['__FIRST AUTHOR'] == author) & (gwas_band['__DATE'] == date) & (gwas_band['__JOURNAL'] == journal) & (gwas_band['__PUBMEDID'] == pubmedid)
        if endpoint == 'CHD':
            final_mask = study_mask & chd_trait_mask
            gwas_band.loc[final_mask, 'is_chd_related'] = True
        elif endpoint == 'istroke':
            final_mask = study_mask & istroke_trait_mask
            gwas_band.loc[final_mask, 'is_istroke_related'] = True
        else:
            raise ValueError(f'Unknown endpoint: {endpoint}')
        gwas_band.loc[study_mask, 'manual_study_selected'] = True
        print(f'[Manual study] {endpoint} | {author} {date} {journal} PMID={pubmedid} | all hits in study = {int(study_mask.sum())} | endpoint-matched hits = {int(final_mask.sum())}')
    helper_cols = [c for c in gwas_band.columns if c.startswith('__')]
    gwas_band = gwas_band.drop(columns=helper_cols)
    return gwas_band
download_cytoband_if_needed(CYTOBAND_FILE)
bands = load_cytobands(CYTOBAND_FILE)
variant = load_variant_positions(VARIANT_CSV)
chrom_sizes = bands.groupby('chrom', as_index=False)['end_mb'].max().rename(columns={'end_mb': 'chrom_len_mb'})
chrom_len_dict = dict(zip(chrom_sizes['chrom'], chrom_sizes['chrom_len_mb']))
snp_chd = load_endpoint_ig('CHD', variant, high_risk_q=HIGH_RISK_Q)
snp_istroke = load_endpoint_ig('istroke', variant, high_risk_q=HIGH_RISK_Q)
snp_chd_band = map_positions_to_bands(snp_chd, bands, chrom_col='chr', pos_col='pos')
snp_istroke_band = map_positions_to_bands(snp_istroke, bands, chrom_col='chr', pos_col='pos')
gwas = pd.read_csv(str(OUTPUT_ROOT / GWAS_CATALOG if (OUTPUT_ROOT / GWAS_CATALOG).exists() else INPUT_ROOT / GWAS_CATALOG), sep='\t', low_memory=False)
gwas.columns = gwas.columns.str.strip()
gwas['CHR_POS'] = pd.to_numeric(gwas['CHR_POS'], errors='coerce')
gwas = gwas.dropna(subset=['CHR_ID', 'CHR_POS']).copy()
gwas['chr'] = gwas['CHR_ID'].apply(normalize_gwas_chr)
gwas = gwas[gwas['chr'].isin(CHR_ORDER)].copy()
gwas['pos'] = gwas['CHR_POS'].astype(int)
print('\nGWAS chromosome counts after normalization:')
print(gwas['chr'].value_counts().sort_index())
gwas_band = map_positions_to_bands(gwas, bands, chrom_col='chr', pos_col='pos')
gwas_band = gwas_band.dropna(subset=['band_id']).copy()
print('\nGWAS mapped cytoband chromosome counts:')
print(gwas_band['chr'].value_counts().sort_index())
gwas_band = apply_manual_study_flags_all_hits(gwas_band=gwas_band, manual_studies=MANUAL_SELECTED_STUDIES)
manual_selected_hits = gwas_band[gwas_band['is_chd_related'] | gwas_band['is_istroke_related']].copy()
manual_selected_hits.to_csv(OUT_SELECTED_STUDIES, index=False)
print('\nSaved manual selected GWAS hits:')
print(OUT_SELECTED_STUDIES)
if not manual_selected_hits.empty:
    summary_group_cols = [c for c in ['is_chd_related', 'is_istroke_related', 'DISEASE/TRAIT', 'STUDY', 'FIRST AUTHOR', 'DATE', 'JOURNAL', 'PUBMEDID'] if c in manual_selected_hits.columns]
    manual_study_summary = manual_selected_hits.groupby(summary_group_cols, dropna=False).agg(n_gwas_hits=('SNPS', 'count') if 'SNPS' in manual_selected_hits.columns else ('DISEASE/TRAIT', 'count'), min_pvalue=('P-VALUE', 'min') if 'P-VALUE' in manual_selected_hits.columns else ('DISEASE/TRAIT', 'count')).reset_index()
    manual_study_summary.to_csv(OUT_SELECTED_STUDY_SUMMARY, index=False)
    print('\nSaved manual selected study summary:')
    print(OUT_SELECTED_STUDY_SUMMARY)
band_table = bands[['band_id', 'chrom', 'band_name', 'start', 'end', 'start_mb', 'end_mb', 'len_bp', 'len_mb']].copy()
all_counts = gwas_band.groupby('band_id', as_index=False).size().rename(columns={'size': 'all_gwas_hits'})
chd_counts = gwas_band[gwas_band['is_chd_related']].groupby('band_id', as_index=False).size().rename(columns={'size': 'chd_gwas_hits'})
istroke_counts = gwas_band[gwas_band['is_istroke_related']].groupby('band_id', as_index=False).size().rename(columns={'size': 'istroke_gwas_hits'})
band_table = band_table.merge(all_counts, on='band_id', how='left')
band_table = band_table.merge(chd_counts, on='band_id', how='left')
band_table = band_table.merge(istroke_counts, on='band_id', how='left')
for col in ['all_gwas_hits', 'chd_gwas_hits', 'istroke_gwas_hits']:
    band_table[col] = band_table[col].fillna(0).astype(int)
band_table = compute_band_enrichment(band_table=band_table, disease_col_hits='chd_gwas_hits', total_col_hits='all_gwas_hits', prefix='chd')
band_table = compute_band_enrichment(band_table=band_table, disease_col_hits='istroke_gwas_hits', total_col_hits='all_gwas_hits', prefix='istroke')
chd_ig_band = snp_chd_band.dropna(subset=['band_id']).groupby('band_id', as_index=False)['mean_abs_ig'].mean().rename(columns={'mean_abs_ig': 'chd_mean_abs_ig'})
istroke_ig_band = snp_istroke_band.dropna(subset=['band_id']).groupby('band_id', as_index=False)['mean_abs_ig'].mean().rename(columns={'mean_abs_ig': 'istroke_mean_abs_ig'})
band_table = band_table.merge(chd_ig_band, on='band_id', how='left')
band_table = band_table.merge(istroke_ig_band, on='band_id', how='left')
band_table['chd_highlight'] = band_table.apply(lambda r: decide_highlight(hit_count=r['chd_gwas_hits'], or_value=r['chd_odds_ratio'], p_value=r['chd_pvalue']), axis=1)
band_table['istroke_highlight'] = band_table.apply(lambda r: decide_highlight(hit_count=r['istroke_gwas_hits'], or_value=r['istroke_odds_ratio'], p_value=r['istroke_pvalue']), axis=1)
band_table['region_type'] = band_table.apply(band_region_type, axis=1)
band_table = band_table.sort_values(['chrom', 'start']).reset_index(drop=True)
band_table.to_csv(OUT_BAND_TABLE, index=False)
snp_band_summary = snp_chd_band[['SNP', 'chr', 'pos', 'mean_abs_ig', 'band_id']].rename(columns={'mean_abs_ig': 'chd_mean_abs_ig'}).merge(snp_istroke_band[['SNP', 'chr', 'pos', 'mean_abs_ig', 'band_id']].rename(columns={'mean_abs_ig': 'istroke_mean_abs_ig'}), on=['SNP', 'chr', 'pos', 'band_id'], how='outer')
snp_band_summary.to_csv(OUT_SNP_TABLE, index=False)
print('Saved band summary with OR:', OUT_BAND_TABLE)
print('Saved SNP-band IG table:', OUT_SNP_TABLE)
chd_q75 = band_table['chd_mean_abs_ig'].dropna().quantile(PLOT_IG_Q)
istroke_q75 = band_table['istroke_mean_abs_ig'].dropna().quantile(PLOT_IG_Q)
band_table['chd_q75_region'] = band_table['chd_mean_abs_ig'].notna() & (band_table['chd_mean_abs_ig'] >= chd_q75)
band_table['istroke_q75_region'] = band_table['istroke_mean_abs_ig'].notna() & (band_table['istroke_mean_abs_ig'] >= istroke_q75)
band_table['any_q75_region'] = band_table['chd_q75_region'] | band_table['istroke_q75_region']
q75_table = band_table[band_table['any_q75_region']].copy()
q75_table.to_csv(OUT_Q75_TABLE, index=False)
plot_chroms = [c for c in CHR_ORDER if c in chrom_len_dict]
print('\nCHD q75 cutoff:', chd_q75)
print('istroke q75 cutoff:', istroke_q75)
print('Chromosomes kept for plotting:', plot_chroms)
print('Saved q75 selected bands:', OUT_Q75_TABLE)
ig_values_all = pd.concat([band_table['chd_mean_abs_ig'], band_table['istroke_mean_abs_ig']], axis=0).dropna()
if len(ig_values_all) > 0:
    vmin = ig_values_all.quantile(0.01)
    vmax = ig_values_all.quantile(0.99)
    if vmin == vmax:
        vmin = ig_values_all.min()
        vmax = ig_values_all.max()
else:
    (vmin, vmax) = (0.0, 1.0)
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = HEATMAP_CMAP
label_region_spans = []
for (chrom, start_band, end_band, label) in LABEL_REGIONS:
    span = get_region_span(bands, chrom, start_band, end_band)
    if span is not None:
        span['label'] = label
        label_region_spans.append(span)
row1_chroms = [f'chr{i}' for i in range(1, 12)]
row2_chroms = [f'chr{i}' for i in range(12, 23)] + ['chrX']
row1_chroms = [c for c in row1_chroms if c in plot_chroms]
row2_chroms = [c for c in row2_chroms if c in plot_chroms]
max_chr_len = max([chrom_len_dict[c] for c in plot_chroms])
row1_y0 = 0
row2_y0 = max_chr_len + ROW_GAP_MB
x_positions = {}
for (i, chrom) in enumerate(row1_chroms):
    x_positions[chrom] = i * COL_GAP
for (i, chrom) in enumerate(row2_chroms):
    x_positions[chrom] = i * COL_GAP
row_y0 = {}
for chrom in row1_chroms:
    row_y0[chrom] = row1_y0
for chrom in row2_chroms:
    row_y0[chrom] = row2_y0
(fig, ax) = plt.subplots(figsize=FIGSIZE)
label_offset_counter = {chrom: 0 for chrom in plot_chroms}
for chrom in plot_chroms:
    if chrom not in x_positions:
        continue
    x0 = x_positions[chrom]
    y_base = row_y0[chrom]
    sub = band_table[band_table['chrom'] == chrom].sort_values('start').copy()
    if sub.empty:
        continue
    chrom_len = chrom_len_dict[chrom]
    chrom_x = x0
    chd_track_x = chrom_x + CHROM_WIDTH + CHROM_TO_TRACK_GAP
    istroke_track_x = chd_track_x + TRACK_WIDTH + TRACK_GAP
    for (_, row) in sub.iterrows():
        y = y_base + row['start_mb']
        h = row['len_mb']
        if row['region_type'] == 'chd':
            face = COLOR_CHD
        elif row['region_type'] == 'istroke':
            face = COLOR_ISTROKE
        elif row['region_type'] == 'both':
            face = COLOR_BOTH
        else:
            face = COLOR_NONE
        ax.add_patch(Rectangle((chrom_x, y), CHROM_WIDTH, h, facecolor=face, edgecolor='white', linewidth=0.12, zorder=2))
    ax.add_patch(Rectangle((chrom_x, y_base), CHROM_WIDTH, chrom_len, facecolor='none', edgecolor=EDGE_COLOR, linewidth=0.75, zorder=3))
    for (_, row) in sub.iterrows():
        y = y_base + row['start_mb']
        h = row['len_mb']
        if row['chd_q75_region'] and pd.notna(row['chd_mean_abs_ig']):
            color_chd = cmap(norm(row['chd_mean_abs_ig']))
        else:
            color_chd = '#FFFFFF'
        if row['istroke_q75_region'] and pd.notna(row['istroke_mean_abs_ig']):
            color_ist = cmap(norm(row['istroke_mean_abs_ig']))
        else:
            color_ist = '#FFFFFF'
        ax.add_patch(Rectangle((chd_track_x, y), TRACK_WIDTH, h, facecolor=color_chd, edgecolor='white', linewidth=0.1, zorder=2))
        ax.add_patch(Rectangle((istroke_track_x, y), TRACK_WIDTH, h, facecolor=color_ist, edgecolor='white', linewidth=0.1, zorder=2))
    ax.add_patch(Rectangle((chd_track_x, y_base), TRACK_WIDTH, chrom_len, facecolor='none', edgecolor=EDGE_COLOR, linewidth=0.55, zorder=3))
    ax.add_patch(Rectangle((istroke_track_x, y_base), TRACK_WIDTH, chrom_len, facecolor='none', edgecolor=EDGE_COLOR, linewidth=0.55, zorder=3))
    ax.text(chrom_x + CHROM_WIDTH / 2, y_base - 8, chrom.replace('chr', ''), ha='center', va='bottom', fontsize=8)
    if chrom in [row1_chroms[0], row2_chroms[0]]:
        ax.text(chd_track_x + TRACK_WIDTH / 2, y_base - 8, 'CHD', ha='center', va='bottom', fontsize=6, rotation=45)
        ax.text(istroke_track_x + TRACK_WIDTH / 2, y_base - 8, 'iStroke', ha='center', va='bottom', fontsize=6, rotation=45)
    spans_this_chrom = [s for s in label_region_spans if s['chrom'] == chrom]
    for span in spans_this_chrom:
        y = y_base + span['start_mb']
        h = span['end_mb'] - span['start_mb']
        box_x = chrom_x - 0.035
        box_w = istroke_track_x + TRACK_WIDTH - chrom_x + 0.07
        ax.add_patch(Rectangle((box_x, y), box_w, h, facecolor='none', edgecolor=LABEL_EDGE_COLOR, linewidth=LABEL_LINEWIDTH, zorder=6))
        k = label_offset_counter[chrom]
        label_offset_counter[chrom] += 1
        text_x = istroke_track_x + TRACK_WIDTH + 0.1 + 0.1 * (k % 2)
        text_y = y + h / 2
        ax.plot([box_x + box_w, text_x - 0.02], [text_y, text_y], color=LABEL_EDGE_COLOR, linewidth=0.65, zorder=6)
        ax.text(text_x, text_y, span['label'], color=LABEL_TEXT_COLOR, fontsize=LABEL_FONTSIZE, ha='left', va='center', rotation=0, zorder=7)
legend_handles = [Patch(facecolor=COLOR_CHD, edgecolor='none', label='Selected CHD study region'), Patch(facecolor=COLOR_ISTROKE, edgecolor='none', label='Selected istroke study region'), Patch(facecolor=COLOR_BOTH, edgecolor='none', label='Shared selected study region'), Patch(facecolor='#FFFFFF', edgecolor=EDGE_COLOR, label='No q75 IG signal'), Patch(facecolor='white', edgecolor=LABEL_EDGE_COLOR, label='Highlighted labeled region')]
ax.legend(handles=legend_handles, frameon=False, fontsize=6.5, title='Regions', title_fontsize=7, loc='upper right', bbox_to_anchor=(1.02, 1.01))
max_cols = max(len(row1_chroms), len(row2_chroms))
x_max = (max_cols - 1) * COL_GAP + CHROM_WIDTH + CHROM_TO_TRACK_GAP + 2 * TRACK_WIDTH + TRACK_GAP + 1.3
y_min = -TOP_LABEL_PAD_MB
y_max = row2_y0 + max_chr_len + BOTTOM_PAD_MB
ax.set_xlim(-0.35, x_max)
ax.set_ylim(y_max, y_min)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
for spine in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(OUTFIG, dpi=300, transparent=True, bbox_inches='tight')
plt.savefig(OUTSVG, dpi=300, transparent=True, bbox_inches='tight')
plt.close()
print('Saved final figure:', OUTFIG)
print('Saved final SVG:', OUTSVG)
(fig_cb, ax_cb) = plt.subplots(figsize=(1.2, 3.2))
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, cax=ax_cb, orientation='vertical')
cb_ticks = np.linspace(vmin, vmax, 3)
cbar.set_ticks(cb_ticks)
cbar.set_ticklabels([f'{x:.4f}' for x in cb_ticks])
cbar.set_label('Mean absolute IG', fontsize=10)
cbar.ax.tick_params(labelsize=8)
plt.savefig(OUT_COLORBAR, dpi=300, transparent=True, bbox_inches='tight')
plt.close(fig_cb)
print('Saved separate colorbar:', OUT_COLORBAR)
print('\nQ75 selected regions:')
print(q75_table[['chrom', 'band_name', 'chd_mean_abs_ig', 'istroke_mean_abs_ig', 'chd_q75_region', 'istroke_q75_region', 'region_type']].to_string(index=False))
print('\nchrX rows in band_table:')
print(band_table.loc[band_table['chrom'] == 'chrX', ['chrom', 'band_name', 'all_gwas_hits', 'chd_gwas_hits', 'istroke_gwas_hits', 'chd_mean_abs_ig', 'istroke_mean_abs_ig', 'region_type']].to_string(index=False))
print('\nLabeled regions in final plot:')
for s in label_region_spans:
    print(f"{s['chrom']} {s['start_band']}-{s['end_band']} ({s['start_mb']:.2f}-{s['end_mb']:.2f} Mb): {s['label']} [included bands: {s['included_bands']}]")
